# Two-band k.p model: Table II, Figure 6 and Table V

The trigonal warping $\kappa$ of the tight-binding models from the Löwdin reduction at $K$ (Table II), the k.p spectra of Fig. 6 and the tensor components of Table V. The reduction, the k.p model and its currents are in the engine. The k.p model with first-principles parameters is in `edge.ipynb`, where it is plotted (Fig. 11).

## Load engine

In [ ]:
%run engine.ipynb

## Settings

In [ ]:
N_kp = 401         # k.p grid across the disk (301 for a quick look; coarser grids ripple)

## Table II: trigonal warping

Band-edge spin block, physical $\lambda$. $v$ in eV Å and $\kappa$ in eV Å$^2$. The last column is the NN closed form (Eq. 11 of the paper), with the spin-orbit shifts in the energies, as a check on the fit.

In [ ]:
print("GGA      v TNN   kappa TNN   kappa NN   kappa NN (closed form)")
for m in materials:
    r_TNN, r_NN = lowdin(m), lowdin(m, model="NN")
    p = params(m, model="NN")
    lam = lambda_SOC[m]
    t1, t2, t11, t12, t22 = p["t1"], p["t2"], p["t11"], p["t12"], p["t22"]
    E_c = p["epsilon1"] - 3 * p["t0"]
    E_v = p["epsilon2"] - 1.5 * (t11 + t22) - 3 * np.sqrt(3) * t12 + lam
    E_u = p["epsilon2"] - 1.5 * (t11 + t22) + 3 * np.sqrt(3) * t12 - lam
    gamma3 = 3 * np.sqrt(2) / 8 * (t2 - np.sqrt(3) * t1) + 9 * np.sqrt(6) / 16 * (t1 - np.sqrt(3) * t2) * (t11 - t22) * (1 / (E_c - E_u) + 1 / (E_v - E_u))
    kappa_closed = p["a"]**2 * gamma3 / 2
    print(f"{m:6s} {r_TNN['v'] * r_TNN['a']:7.3f} {r_TNN['kappa'] * r_TNN['a']**2:+10.4f} {r_NN['kappa'] * r_NN['a']**2:+10.4f} {kappa_closed:+12.4f}")

for model in ["TNN", "NN"]:
    kappa_LDA = []
    for m in materials:
        r = lowdin(m, "LDA", model)
        kappa_LDA.append(r["kappa"] * r["a"]**2)
    print(f"LDA kappa {model}:", np.round(kappa_LDA, 4))

## Figure 6: k.p spectra

Parameters from the reduction of the MoS$_2$ TNN model at $\lambda = 0$, with $\lambda$ put back into $d_z$. $k$ in $1/a$, disk of radius $0.6/a$, $\nu = 10$ meV. One valley at $B = 0$, both valleys at $B = 0$ (the injection current cancels), and both valleys at 30 T. A few minutes.

In [ ]:
kp = lowdin("MoS2", lam=0.0)
kp["lam"] = lambda_SOC["MoS2"]
a = kp["a"]
print(f"MoS2 k.p: v = {kp['v'] * a:.3f} eV A, kappa = {kp['kappa'] * a**2:.4f} eV A^2, w = {kp['w'] * a**2:.3f} eV A^2, m_D = {kp['mD']:.4f} eV")

Eg_kp = 2 * kp["mD"] - kp["lam"]
omega_kp = np.linspace(Eg_kp - 0.1, Eg_kp + 1.0, 221)

file = f"data/kp_spectra_N{N_kp}.npz"
if not os.path.exists(file):
    out = {}
    for tag, b, pairs in [("K_B0", 0.0, [(1, 1)]), ("KKp_B0", 0.0, [(1, 1), (-1, -1)]), ("KKp_B30", mu_B * 30, [(1, 1), (-1, -1)])]:
        eta, sigma = kp_currents(kp, b, omega_kp, 0.01, 0.6, N_kp, pairs, shift=True)
        out["eta_" + tag] = eta_SI(a) * eta[:, 0, 0, 0]
        out["shift_" + tag] = shift_SI(a) * sigma[:, 1, 1, 1]
    np.savez(file, **out)
spectra = np.load(file)

In [ ]:
shown = omega_kp <= Eg_kp + 0.45
fig, axes = plt.subplots(1, 2, figsize=(TW, 2.45), gridspec_kw=dict(wspace=0.42))
ax = axes[0]
ax.plot(omega_kp[shown], spectra["eta_K_B0"][shown], "--", color=BLUE, label=r"$K$, $B=0$ (left)")
ax.plot(omega_kp[shown], spectra["eta_KKp_B0"][shown], color=GREY, label=r"$K+K'$, $B=0$ (left)")
ax2 = ax.twinx()
ax2.plot(omega_kp[shown], 1e3 * spectra["eta_KKp_B30"][shown], color=RED, label=r"$K+K'$, $B=30$ T (right)")
ax.set_ylabel(r"$\eta^{x;xx}$ ($%s$)" % UNIT)
ax2.set_ylabel(r"$\eta^{x;xx}_{K+K'}$ ($10^{-3}\,%s$)" % UNIT, color=RED)
ax2.tick_params(axis="y", which="both", colors=RED)
ax2.spines["right"].set_color(RED)
ax.tick_params(right=False, which="both")
for axis, key, scale in ((ax, "eta_K_B0", 1), (ax2, "eta_KKp_B30", 1e3)):
    biggest = np.abs(spectra[key][shown] * scale).max()
    axis.set_ylim(-0.05 * biggest, 1.42 * biggest)
h1, l1 = ax.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, loc="upper left")
ax = axes[1]
ax.plot(omega_kp[shown], spectra["shift_K_B0"][shown], "--", color=BLUE, label=r"$K$, $B=0$")
ax.plot(omega_kp[shown], spectra["shift_KKp_B0"][shown], color=GREY, lw=2.2, label=r"$K+K'$, $B=0$")
ax.plot(omega_kp[shown], spectra["shift_KKp_B30"][shown], color=RED, lw=0.9, label=r"$K+K'$, $B=30$ T")
ax.set_ylabel(r"$\sigma^{y;yy}$ ($%s$)" % SUNIT)
ax.legend(loc="lower right")
ax.set_ylim(bottom=0)
for i, ax in enumerate(axes):
    zero(ax)
    ax.set_xlabel(r"$\hbar\omega$ (eV)")
    label(ax, f"({'ab'[i]})")
plt.savefig("figures/kp_spectra.pdf")
plt.show()

ratio = np.abs(spectra["eta_KKp_B30"][shown]).max() / np.abs(spectra["eta_K_B0"][shown]).max()
print(f"The K + K' peak at 30 T is {100 * ratio:.2f}% of the single-valley peak.")

## Table V: tensor components at $E_g + 0.1$ eV

The same model at one photon energy, with the components the mirror forbids at $B = 0$.

In [ ]:
omega_V = np.array([Eg_kp + 0.1])
print("               eta^x;xx (A m V^-2 s^-1)  eta^y;yy    sigma^y;yy (nm uA V^-2)  sigma^x;xx")
for tag, b, pairs in [("K+K', B = 0 ", 0.0, [(1, 1), (-1, -1)]), ("K+K', B = 30 T", mu_B * 30, [(1, 1), (-1, -1)]), ("K, B = 0     ", 0.0, [(1, 1)])]:
    eta, sigma = kp_currents(kp, b, omega_V, 0.01, 0.6, N_kp, pairs, shift=True)
    print(f"{tag} {eta_SI(a) * eta[0, 0, 0, 0]:+.3e}  {eta_SI(a) * eta[0, 1, 1, 1]:+.3e}  {shift_SI(a) * sigma[0, 1, 1, 1]:.3f}  {shift_SI(a) * sigma[0, 0, 0, 0]:+.3e}")